# Lab 5 (Bonus) — Reranking: Precision After Recall

**Bonus lab.** Labs 1–3 built a retriever; Lab 4 wired it to an LLM and an agent. This bonus lab adds the one stage we only *gestured* at in Lab 3: a **reranker** — a second-pass model that re-scores your top candidates for precision.

## What you'll learn
- **What reranking is** and why it's a *second stage* on top of retrieval (recall → precision)
- The two architecturally different kinds: **pointwise (cross-encoder)** vs **listwise**
- How to call the **rerank inference API directly** (`POST _inference/rerank/<id>`)
- **Jina Reranker v2 vs v3** head-to-head on the same candidates
- A **decision matrix**: when to use a reranker at all, and pointwise vs listwise
- The production path: the **`text_similarity_reranker`** retriever, in one `_search`

## Before you start
- **In Instruqt:** credentials are pre-configured — just run the cells.
- **Re-running from the repo:** `export ES_ENDPOINT=https://...` and `export ES_API_KEY=...`

> Everything here uses the **same** Elastic Inference Service you've used all workshop — the rerank models are built-in endpoints, no extra setup.

In [ ]:
# --- Workshop helpers (inline — same block across all notebooks, plus rerank helpers) ---

import os, json
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

# Reranker endpoints (both built-in on the provisioned project)
POINTWISE_ID = ".jina-reranker-v2-base-multilingual"   # cross-encoder, scores each doc alone
LISTWISE_ID  = ".jina-reranker-v3"                      # listwise, scores the whole set jointly

def r_semantic(query):
    return {"standard": {"query": {"semantic": {"field": "body_semantic", "query": query}}}}

def r_bm25(query):
    return {"standard": {"query": {"multi_match": {
        "query": query, "fields": ["title^3", "body"], "type": "best_fields"}}}}

def r_rrf(query, rank_constant=60, rank_window_size=100):
    return {"rrf": {"retrievers": [r_bm25(query), r_semantic(query)],
                    "rank_constant": rank_constant, "rank_window_size": rank_window_size}}

def get_candidates(query, retriever_builder=r_rrf, size=8):
    """Run a retriever and return flat dicts WITH body text — the input a reranker needs."""
    resp = es.search(index=INDEX, retriever=retriever_builder(query), size=size,
                     source=["id", "title", "trap_type", "body"])
    return [{"id": h["_source"]["id"], "title": h["_source"]["title"],
             "trap_type": h["_source"].get("trap_type"), "body": h["_source"]["body"]}
            for h in resp["hits"]["hits"]]

def docs_by_id(ids):
    """Fetch specific docs by id (used to hand-build a candidate set)."""
    resp = es.mget(index=INDEX, ids=ids, source=["id", "title", "trap_type", "body"])
    return [{"id": d["_source"]["id"], "title": d["_source"]["title"],
             "trap_type": d["_source"].get("trap_type"), "body": d["_source"]["body"]}
            for d in resp["docs"] if d.get("found")]

def rerank(inference_id, query, docs, text_field="body"):
    """Call the EIS rerank endpoint directly (POST _inference/rerank/<id>).
    Returns a NEW list of the same docs, reordered by relevance_score (desc),
    each annotated with a 'rerank_score'. The API returns results pre-sorted."""
    inputs = [d[text_field] for d in docs]
    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/rerank/{inference_id}",
        headers={"Authorization": f"ApiKey {ES_API_KEY}", "Content-Type": "application/json"},
        json={"query": query, "input": inputs},
        timeout=60,
    )
    resp.raise_for_status()
    out = []
    for r in resp.json()["rerank"]:          # [{"index": i, "relevance_score": s}, ...]
        d = dict(docs[r["index"]])
        d["rerank_score"] = r["relevance_score"]
        out.append(d)
    return out

def show_ranked(docs, score_key=None, fields=("id", "title", "trap_type")):
    """Print a ranked list. If score_key is set, show that score column."""
    if not docs:
        print("  (none)"); return
    for rank, d in enumerate(docs, 1):
        cols = "  ".join(str(d.get(f, "") or "") for f in fields)
        s = f"  {d[score_key]:.4f}" if score_key and score_key in d else ""
        print(f"  #{rank:<2}{s}  {cols}")

info = es.info()
count = es.count(index=INDEX)["count"]
print(f"Connected to ES {info['version']['number']} | {count} docs in '{INDEX}'")
print("✓ Helpers loaded (POINTWISE_ID, LISTWISE_ID, rerank(), get_candidates(), docs_by_id())")

## What reranking *is* — a second stage on top of retrieval

Everything you built in Labs 1–3 is **bi-encoder** retrieval: the model encodes the query and each document *independently* into vectors, and you compare those vectors (cosine similarity / RRF over ranks). That's what makes it fast enough to search millions of documents — every document vector is computed once, ahead of time, at index.

A **reranker** works differently. It reads the **query and a candidate document *together*** and scores how relevant that specific document is to that specific query. Because it sees both at once, it can catch relevance that vector proximity misses — but it can't be precomputed, so it's far too expensive to run over a whole corpus.

So you use it as a **second stage**:

```
            STAGE 1 — RECALL                      STAGE 2 — PRECISION
   ┌─────────────────────────────┐        ┌──────────────────────────────┐
   │  hybrid retriever (RRF)      │        │  reranker reads (query, doc)  │
   │  fast, scans the whole index │  top-N │  pairs and re-scores just     │
   │  → top 50–100 candidates     │ ─────► │  those N → returns best K      │
   └─────────────────────────────┘        └──────────────────────────────┘
        cheap, approximate                     expensive, precise
```

**Stage 1 (recall)** casts a wide, cheap net — get the right documents *somewhere* in the top 50. **Stage 2 (precision)** spends real compute only on those 50 to get the order at the very top exactly right. That top-K is what feeds a RAG prompt or an agent — so its precision is what the user actually feels.

## The two TYPES of reranker — pointwise vs listwise

Not all rerankers work the same way. There are two architectures, and the difference matters:

**Pointwise (cross-encoder) — e.g. Jina Reranker v2**
Scores each candidate **on its own**: one `(query, document)` forward pass per document. Document #3 is scored with no knowledge that document #4 exists. Simple, robust, trivially parallel — this is the classic cross-encoder reranker.

**Listwise — e.g. Jina Reranker v3**
Puts the **whole candidate set into one context window** and scores them **jointly**, so the model can compare documents *against each other* in a single pass ("last but not late" interaction). When two candidates are near-duplicates or overlap heavily, a listwise model can see that and order them sensibly — a pointwise model literally cannot, because it never sees them together. (Jina v3: listwise, GA Oct 2025, ~0.6B params, reranks up to ~64 docs per call.)

| | **Pointwise (cross-encoder, Jina v2)** | **Listwise (Jina v3)** |
|---|---|---|
| Scores each doc… | **independently**, one pair at a time | **jointly**, whole set in one pass |
| Sees other candidates? | ❌ no | ✅ yes (document↔document interaction) |
| Best at | clean, distinct candidates | similar / near-duplicate / overlapping candidates |
| Scales by | per-doc (batchable, no cap) | one bigger call (candidate cap, ~64) |

Elastic exposes **both** as built-in EIS endpoints — you swap a single `inference_id` to switch. We'll run them head-to-head in a moment, then give a full *which-to-choose* matrix once you've seen the difference.

In [ ]:
# Confirm both rerank endpoints exist on this project (same inspection skill as Lab 1).
all_eps = es.inference.get()
print("Rerank endpoints available (task_type='rerank'):\n")
for ep in all_eps.get("endpoints", []):
    if ep.get("task_type") == "rerank":
        print(f"  {ep.get('inference_id'):<40} service={ep.get('service')}")

print(f"\nWe'll use:\n  pointwise (v2): {POINTWISE_ID}\n  listwise  (v3): {LISTWISE_ID}")

## Calling the reranker directly — the rawest form

Before the polished retriever, let's see the actual API a reranker exposes. It's dead simple:

```
POST _inference/rerank/<inference_id>
{
  "query": "encrypt traffic between cluster nodes",
  "input": [ "full text of doc A", "full text of doc B", ... ]
}
```

and it returns the candidates **re-sorted by relevance**, each with a score:

```
{ "rerank": [ {"index": 3, "relevance_score": 0.94},
              {"index": 0, "relevance_score": 0.71}, ... ] }
```

`index` points back into the `input` list you sent. Our `rerank()` helper (loaded above) wraps exactly this call and re-attaches each score to the original document. This is the *same* call the `text_similarity_reranker` retriever makes under the hood — we're just doing it by hand first so there's no magic.

In [ ]:
# Recall stage (RRF) → then rerank that candidate set and compare the order.
query = "encrypt traffic between Elasticsearch nodes"   # target: doc-010 (TLS for cluster comms)

candidates = get_candidates(query, retriever_builder=r_rrf, size=8)

print(f"QUERY: {query!r}\n")
print("STAGE 1 — RRF recall order (bi-encoder + BM25, no reranker):")
show_ranked(candidates)

reranked = rerank(LISTWISE_ID, query, candidates)
print(f"\nSTAGE 2 — after reranking with {LISTWISE_ID}:")
show_ranked(reranked, score_key="rerank_score")

print("\nThe reranker read each candidate's body *together with the query* and re-scored it.")
print("Watch whether the doc you'd actually want (TLS / cluster comms) moves up vs the RRF order.")

## Pointwise vs listwise — head to head

Now the part that's hard to see in the abstract: does the **listwise** model actually behave differently from the **pointwise** one? The honest answer is *it depends on the candidates*. The difference shows up when the set contains documents that are **similar to each other** — because that's the only situation where "seeing the other candidates" can change a decision.

So we deliberately build a candidate set with a **near-duplicate pair**:
- **doc-009** — *Set up security in self-managed Elasticsearch deployments*
- **doc-010** — *TLS encryption for cluster communications*

Both are about securing a cluster; they overlap heavily. For a query specifically about **encrypting traffic between nodes**, doc-010 is the sharper answer. We add a couple of distractors (container-exit / crash docs) as noise.

> **Set expectations honestly:** on a small, clean candidate set the two models often *agree* — and that's a perfectly valid result to observe. The listwise advantage grows with **more candidates** and **more overlap**. Watch the **order of the two near-duplicates** and their score *gap*.

In [ ]:
# Same query, same candidates — pointwise (v2) vs listwise (v3), side by side.
query = "how do I encrypt traffic between cluster nodes"

# Hand-built set: the near-duplicate pair + distractors (so there's something to disagree about).
candidate_ids = ["doc-010", "doc-009", "doc-061", "doc-062", "doc-001"]
candidates = docs_by_id(candidate_ids)

print(f"QUERY: {query!r}")
print(f"Candidates: {[d['id'] for d in candidates]}  (doc-009/doc-010 are near-duplicates)\n")

def safe_rerank(label, inference_id):
    try:
        ranked = rerank(inference_id, query, candidates)
        print(f"{label}  ({inference_id}):")
        show_ranked(ranked, score_key="rerank_score")
    except Exception as e:
        print(f"{label}  ({inference_id}): ⚠ unavailable — {str(e)[:120]}")
    print()

safe_rerank("POINTWISE v2", POINTWISE_ID)
safe_rerank("LISTWISE  v3", LISTWISE_ID)

print("Compare the two orderings — especially where doc-009 and doc-010 land relative to each\n"
      "other, and the score gap between them. If they agree here, that's expected on a clean\n"
      "5-doc set; the listwise edge is bigger on larger, noisier candidate pools.")

## Which reranker should you choose — pointwise vs listwise?

Now that you've seen them run, here's how to pick. This is about **pointwise vs listwise**; the next cell covers the separate question of whether to add a reranker *at all*.

| | **Pointwise (cross-encoder, Jina v2)** | **Listwise (Jina v3)** |
|---|---|---|
| **How it scores** | each `(query, doc)` **independently** | the whole candidate set **jointly**, one pass |
| **Choose it when** | candidates are already distinct; you stream/score docs in isolation; you want a stable per-doc score you can cache or threshold | candidates are **similar / near-duplicate / overlapping** and the order *among them* matters; it's the final top-k feeding a RAG prompt or agent |
| **Why *not*** | can't see that two candidates are near-dupes, so it may mis-order them relative to each other | one call must hold the whole set in its context window (cap ~64 docs); the per-doc score isn't independent, so it's harder to cache/threshold |
| **Cost / latency** | scales per document; trivially parallel and batchable | one larger call; bounded candidates per call |
| **Maturity** | the classic, battle-tested cross-encoder | newer (GA 2025-10), SOTA on noisy/ambiguous sets |

**Why choose listwise (v3):** cross-document context lets it resolve ties between near-duplicates that a pointwise model *structurally cannot see*. It tends to win exactly where retrieval is hardest — lots of similar candidates competing for the top slots.

**Why *not* listwise:** if your candidates are already clean and distinct, it's extra capability you don't need; the candidate-window cap and single-call shape are also a worse fit if you want to score documents independently (streaming, caching, per-doc thresholds).

**Why choose pointwise (v2):** simplicity, independence, batchability, and a long track record. A per-doc score you can compute, cache, and threshold in isolation.

**Default guidance:** start with **pointwise**. Reach for **listwise** when you *observe* near-duplicate or ambiguous candidates fighting for the top — which is precisely what you can measure with the judgment-set / MRR harness you built in Lab 3.

## The production path — `text_similarity_reranker` in one query

Calling the rerank API by hand was for understanding. In production you don't fetch candidates, marshal their text, POST them, and re-sort yourself — you let Elasticsearch do the whole **recall → precision** pipeline in a single `_search`:

```
text_similarity_reranker
├── retriever:         <your RRF hybrid>     ← STAGE 1: recall (Lab 3)
├── field:             "body"                 ← which field the reranker reads
├── inference_id:      ".jina-reranker-v3"    ← STAGE 2: which reranker (swap v2/v3 here)
├── inference_text:    "<the query>"          ← the query the reranker scores against
└── rank_window_size:  50                     ← how many candidates stage 1 hands to stage 2
```

It fetches `rank_window_size` candidates from the inner retriever, reranks them with the named endpoint, and returns the reordered top hits — same pattern as the Lab 3 reranker cell, now with a one-line swap between **pointwise** and **listwise**.

In [ ]:
# Production recall->precision in a single _search. Guarded: if the endpoint is missing on
# this project, the cell explains rather than crashes (same gating style as Lab 3).
query = "how do I secure traffic between nodes"   # target doc-010 (TLS for cluster comms)
RERANKER_ID = LISTWISE_ID                          # swap to POINTWISE_ID to compare

reranker_retriever = {
    "text_similarity_reranker": {
        "retriever": r_rrf(query),       # STAGE 1: recall (the Lab 3 hybrid retriever)
        "field": "body",                  # field the reranker scores
        "inference_id": RERANKER_ID,      # STAGE 2: which reranker
        "inference_text": query,          # REQUIRED: the query text
        "rank_window_size": 50,           # candidates passed from stage 1 to stage 2
    }
}

def show_search(resp, fields=("id", "title", "trap_type")):
    for rank, h in enumerate(resp["hits"]["hits"], 1):
        src = h.get("_source", {})
        cols = "  ".join(str(src.get(f, "") or "") for f in fields)
        sc = f"  {h['_score']:.4f}" if h.get("_score") is not None else ""
        print(f"  #{rank:<2}{sc}  {cols}")

try:
    print(f"STAGE 1 — RRF recall: {query!r}")
    show_search(es.search(index=INDEX, retriever=r_rrf(query), size=5,
                          source=["id", "title", "trap_type"]))
    print(f"\nSTAGE 1+2 — text_similarity_reranker ({RERANKER_ID}):")
    resp = es.search(index=INDEX, retriever=reranker_retriever, size=5,
                     source=["id", "title", "trap_type"])
    show_search(resp)
    print("\n(scores are now cross-encoder/listwise relevance scores, not RRF rank-fusion scores)")
except Exception as e:
    print(f"⚠ Reranker unavailable: {str(e)[:160]}")
    print(f"  Verify '{RERANKER_ID}' appears in es.inference.get() (task_type='rerank').")
    print("  The RRF result above is still strong on its own.")

## When to add a reranker *at all* — and when to skip it

Choosing pointwise vs listwise (above) only matters *if* you add a reranker. Here's whether to:

**Add a reranker when:**
- You have a **latency budget** for a second pass and **precision matters more than raw throughput**.
- Stage 1 returns **many plausible candidates** and the exact order of the top few matters.
- It's the **final top-k feeding a RAG prompt or an agent** — where a wrong #1 becomes a wrong answer (exactly the Lab 4 lesson: retrieval quality bounds answer quality).

**Skip it when:**
- Stage 1 is already crisp (the right doc is reliably #1) — there's nothing to fix. **On our 62-doc corpus this is usually the case**, which is why these cells are about *mechanics*, not a dramatic accuracy jump.
- Latency is tight and you can't afford the extra model call.
- The candidate set is tiny — a reranker can even shuffle a lexically-similar distractor upward.

**Sizing:** rerank a **window** (top 50–100), never the whole index — that's the entire point of the two-stage split. Bigger windows = more recall fed to precision, but more rerank cost per query.

**Tie-back to Lab 4:** the cleanest place to drop a reranker is right before you build the prompt — feed the Agent Builder hybrid tool's results through a reranker and the agent reasons over a sharper top-k.

## Wrap-up

You added the **precision stage** that sits on top of everything from Labs 1–3:

- **Reranking = stage 2.** Recall (fast, wide, bi-encoder/RRF) → precision (expensive, narrow, reranker reads query+doc together).
- **Two kinds.** *Pointwise* cross-encoders (Jina v2) score each doc alone; *listwise* (Jina v3) scores the whole candidate set jointly and shines on near-duplicate/ambiguous sets.
- **Both are built-in EIS endpoints** — swap one `inference_id`. You called the rerank API directly *and* via the `text_similarity_reranker` retriever.
- **It's a tool, not a default.** Add it for the final top-k when precision matters; skip it when stage 1 is already sharp.

The retriever you built across the workshop is still the foundation. Reranking is the optional, high-precision layer you reach for when the order at the very top has to be right — like the top-k feeding an agent.

---
*That's the bonus. Thanks for staying — go build something.*